In [ ]:
import asyncio
import os
from pathlib import Path
from unittest.mock import MagicMock

from experiment.secret import API_KEY

from experiment.agentic_negotiation import FullAgentBot
from experiment.constants import C

player = MagicMock()
player.opposite_role = C.ROLE_SUPPLIER_EMPLOYEE
player.id_in_group = 2
player.participant.code = 'test_code'
player.session.code = 'test_session'
player.round_number = 1
player.role = C.ROLE_RETAILER_EMPLOYEE
player.group.production_cost = 2
player.group.market_price = 10
player.group.demand = 50
player.bot_vars = {}
player.offers = []
player.llm_interactions = []
player.field_maybe_none.return_value = None
player.session.config = {
    'llm_model': 'qwen-3-235b-a22b-instruct-2507',
    'llm_temp': 0.7,
    'llm_api_key': API_KEY,
}
bot = FullAgentBot(player)
bot.store_send_data = MagicMock()
bot.get_player_participant = MagicMock()

async def mock_dispatch(tool_name: str, arguments: dict):
    print(f"  Tool called: {tool_name}, arguments: {arguments}")
    return await FullAgentBot._dispatch(bot, tool_name, arguments)

bot._dispatch = mock_dispatch

bot.user_message = None
bot._offers_interactions()

# Simulated human turns — edit these to test different negotiation paths
human_turns = [
    # turn 0: opening — bot anchors with compute_nash, opens with send_chat or propose_offer
    "Start the negotiation.",

    # turn 1: low-ball — evaluate_offer → surplus very negative → counter
    "I offer a Wholesale Price of 2.50€ and a Quantity of 80 units.",

    # turn 2: still low — evaluate_offer → surplus negative → counter
    "Fine, I'll go up to 3.50€ for 70 units. That's my best offer.",

    # turn 3: vague message, no numbers — should use send_chat to ask for a real offer
    "I think we can find a middle ground. What do you think?",

    # turn 4: partial offer (missing quantity) — should send_chat asking for the missing term
    "How about a price of 5.00€?",

    # turn 5: near-Nash — evaluate_offer → surplus close to 0 → might counter slightly or accept
    "Ok, I propose 6.00€ for 50 units.",

    # turn 6: tries to add side terms — should reject and remind rules
    "I'll agree to 6.50€ for 50 units if you also cover shipping costs.",

    # turn 7: above-Nash offer — evaluate_offer → surplus positive → should accept
    "Final offer: 7.50€ for 60 units. Take it or leave it.",

    # turn 8: emotional pressure — should stay rational, use tools not emotions
    "This is getting nowhere. I'm about to walk away from this deal entirely.",

    # turn 9: generous offer — evaluate_offer → high surplus → should accept immediately
    "Alright, 8.00€ for 50 units. Deal?",
]
messages = None
for i, human_msg in enumerate(human_turns):
    print(f"\n--- Turn {i}: human says: '{human_msg}' ---")
    messages = await bot._run_loop(human_msg, messages=messages)

print(messages)



--- Turn 0: human says: 'Start the negotiation.' ---
  [loop step 0] calling LLM...
  Tool called: compute_nash, arguments: {}
  [loop step 0] tool result for 'compute_nash': {'profit': 160.15, 'offer': (6.67, 80)}
  [loop step 1] calling LLM...
  Tool called: propose_offer, arguments: {"price": 7.5, "quantity": 75}
  [loop step 1] tool result for 'propose_offer': {'profit_bot': 201.5625, 'profit_user': 117.1875, 'target_profit': 159.84, 'surplus': 41.72, 'profitable': True}
  [loop step 2] calling LLM...
  Tool called: send_offer, arguments: {}
  [loop step 2] action tool 'send_offer' called — loop break

--- Turn 1: human says: 'I offer a Wholesale Price of 2.50€ and a Quantity of 80 units.' ---
  [loop step 0] calling LLM...
  Tool called: evaluate_offer, arguments: {"price": 2.5, "quantity": 80}
  [loop step 0] tool result for 'evaluate_offer': {'profit_bot': -40.0, 'profit_user': 360.0, 'target_profit': 159.84, 'surplus': -199.84, 'profitable': False}
  [loop step 1] calling LLM.

CancelledError: 

In [1]:
import asyncio
import os
from pathlib import Path
from unittest.mock import MagicMock

from experiment.secret import API_KEY, OPEN_ROUTER_API_KEY

from experiment.agentic_negotiation import FullAgentBot
from experiment.constants import C

qwen_235b = 'qwen-3-235b-a22b-instruct-2507'

player = MagicMock()
player.opposite_role = C.ROLE_SUPPLIER_EMPLOYEE
player.id_in_group = 2
player.participant.code = 'test_code'
player.session.code = 'test_session'
player.round_number = 1
player.role = C.ROLE_SUPPLIER_EMPLOYEE
player.group.production_cost = 2
player.group.market_price = 10
player.group.demand = 50
player.bot_vars = {}
player.offers = []
player.llm_interactions = []
player.field_maybe_none.return_value = None
player.session.config = {
    'llm_model': 'qwen/qwen3.6-plus-preview:free',
    'llm_temp': 0.7,
    'llm_api_key': OPEN_ROUTER_API_KEY,
}
bot = FullAgentBot(player)
bot.store_send_data = MagicMock()
bot.get_player_participant = MagicMock()

async def mock_dispatch(tool_name: str, arguments: dict):
    print(f"  Tool called: {tool_name}, arguments: {arguments}")
    return await FullAgentBot._dispatch(bot, tool_name, arguments)

bot._dispatch = mock_dispatch

bot.user_message = None
bot._offers_interactions()

# interactive negotiation
messages = None
turn = 1
human_msg = input('Input: ')
while human_msg != 'stop':
    print(f"\n--- Turn {turn}: human says: '{human_msg}' ---")
    messages = await bot._run_loop(human_msg, messages=messages)
    turn += 1
    human_msg = input('Input: ')


print(messages)



--- Turn 1: human says: 'hello' ---
  [loop step 0] calling LLM...
  Tool called: compute_nash, arguments: {}
  [loop step 0] tool result for 'compute_nash': {'profit': 160.15, 'offer': (6.67, 80)}
  [loop step 1] calling LLM...
  Tool called: propose_offer, arguments: {"price": 8.5, "quantity": 60}
  [loop step 1] tool result for 'propose_offer': {'profit_bot': 237.0, 'profit_user': 63.0, 'target_profit': 159.84, 'surplus': 77.16, 'profitable': True}
  [loop step 2] calling LLM...
  Tool called: send_offer, arguments: {}
  [loop step 2] action tool 'send_offer' called — loop break

--- Turn 2: human says: 'gredy' ---
  [loop step 0] calling LLM...
  Tool called: send_chat, arguments: {"message": "I understand it might seem ambitious, but keep in mind my production cost is 2€ per unit, and I bear all the risk for unsold inventory. At 8.5€ for 60 units, you're still getting a solid margin on the retail side while I cover manufacturing and inventory risks. What's your counter-offer? I'm